# 4장 실습 — 시간대별 속도가 경로를 바꿉니다

3장에서는 도로마다 속도가 하나였습니다. 실제로는 시간대마다 다릅니다.
속도를 바꾸면 소요시간만 늘어나는 것이 아니라 **경로 자체가 달라집니다.**
교재 4장에 대응합니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 속도 컬럼이 여러 개입니다 (교재 4.1)

In [ ]:
import pandas as pd

from smartmob.data import data_path

edges = pd.read_parquet(data_path("hanam/road_graph_edges.parquet"))
speed_cols = [c for c in edges.columns if c.endswith("_p50") or c.endswith("speed_kmh")]
speed_cols

`free_flow_speed_kmh` 는 막히지 않을 때의 속도이고, `_p50` 이 붙은 것은
그 시간대 실측 속도의 중앙값입니다.

In [ ]:
summary = edges[speed_cols].describe().T[["mean", "50%", "max"]]
summary.round(1)

자유류 속도가 가장 빠르고, 출퇴근 시간대가 가장 느립니다.

## 2. 시간과 속도 컬럼의 대응 (교재 4.2)

몇 시가 어느 컬럼을 쓰는지는 `HOUR_TO_COLUMN` 에 적혀 있습니다.

In [ ]:
from smartmob.teaching.eta import HOUR_TO_COLUMN

for hour in [3, 8, 12, 18, 21]:
    print(f"{hour:2d}시 → {HOUR_TO_COLUMN[hour]}")

## 3. 같은 구간, 다른 시간 (교재 4.2)

하남시청에서 미사역까지를 시간대별로 구해 봅니다.

In [ ]:
from smartmob.data import load_road_graph
from smartmob.teaching.dijkstra import shortest_path

HANAM_CITY_HALL = (37.5393, 127.2148)
MISA_STATION = (37.5606, 127.1930)

HOURS = [0, 6, 8, 11, 14, 17, 19, 22]
rows = []
for hour in HOURS:
    g = load_road_graph("hanam", modes=("drive",), speed_column=HOUR_TO_COLUMN[hour])
    p = shortest_path(g, HANAM_CITY_HALL, MISA_STATION, algorithm="dijkstra")
    rows.append({
        "hour": hour,
        "컬럼": HOUR_TO_COLUMN[hour],
        "소요_분": round(p.duration_s / 60, 2),
        "거리_km": round(p.distance_km(g), 2),
        "노드수": len(p.nodes),
    })

curve = pd.DataFrame(rows)
curve

## 4. 그림으로 봅니다

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve["hour"], curve["소요_분"], marker="o", color="#4C6EF5")
ax.set_xlabel("출발 시각 (시)")
ax.set_ylabel("소요시간 (분)")
ax.set_title("하남시청 → 미사역, 시간대별 소요시간")
ax.grid(alpha=0.3)
plt.tight_layout();

## 5. 경로 자체가 달라집니다 (교재 4.3)

소요시간만 늘어나는 것이 아닙니다. 막히는 큰길을 피해 다른 길로 돌아갑니다.
새벽 3시와 저녁 6시의 경로가 같은지 봅니다.

In [ ]:
g_night = load_road_graph("hanam", modes=("drive",), speed_column=HOUR_TO_COLUMN[3])
g_peak = load_road_graph("hanam", modes=("drive",), speed_column=HOUR_TO_COLUMN[18])

p_night = shortest_path(g_night, HANAM_CITY_HALL, MISA_STATION, algorithm="dijkstra")
p_peak = shortest_path(g_peak, HANAM_CITY_HALL, MISA_STATION, algorithm="dijkstra")

banner("새벽 3시 vs 저녁 6시")
print(f"새벽  {p_night.duration_s / 60:5.2f}분, 노드 {len(p_night.nodes)}개")
print(f"저녁  {p_peak.duration_s / 60:5.2f}분, 노드 {len(p_peak.nodes)}개")
print(f"경로가 같은가: {p_night.nodes == p_peak.nodes}")

shared = len(set(p_night.nodes) & set(p_peak.nodes))
print(f"겹치는 노드 {shared}개 / 새벽 경로 {len(p_night.nodes)}개")

경로가 다르다면, 그것이 시간대별 속도를 넣는 이유입니다.
하나의 평균 속도로 시뮬레이션하면 이 차이가 통째로 사라집니다.

## 6. 지도에 두 경로를 겹쳐 봅니다

In [ ]:
night_xy = p_night.coords(g_night)
peak_xy = p_peak.coords(g_peak)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([c[1] for c in night_xy], [c[0] for c in night_xy],
        lw=3, alpha=0.7, label="새벽 3시")
ax.plot([c[1] for c in peak_xy], [c[0] for c in peak_xy],
        lw=2, ls="--", label="저녁 6시")
ax.scatter(*HANAM_CITY_HALL[::-1], c="black", zorder=5, label="하남시청")
ax.scatter(*MISA_STATION[::-1], c="crimson", zorder=5, label="미사역")
ax.set_xlabel("경도")
ax.set_ylabel("위도")
ax.legend()
ax.set_title("시간대에 따라 달라지는 경로")
plt.tight_layout();

## 7. 빈칸

### 7.1 가장 느린 시간대

위 `curve` 표에서 소요시간이 가장 긴 시각과 가장 짧은 시각을 찾고, 그 비율을 구합니다.

In [ ]:
slowest_hour = None     # 가장 오래 걸리는 출발 시각 (시)
fastest_hour = None     # 가장 빨리 가는 출발 시각 (시)
ratio = None            # 느린 쪽 소요시간 ÷ 빠른 쪽 소요시간

banner("빈칸 7.1")
todo("가장 느린 시각", slowest_hour)
todo("가장 빠른 시각", fastest_hour)
todo("비율", ratio, fmt=lambda v: f"{v:.2f}배")

### 7.2 다른 구간에서도 그런가

하남 안의 다른 두 지점을 골라 같은 곡선을 그려 봅니다.
좌표는 `G.coord` 에서 아무 노드나 두 개 골라도 되고, 지도에서 찾아 넣어도 됩니다.

시간대별 차이가 위 구간보다 큰지 작은지, 왜 그런지 두 줄로 적습니다.

In [ ]:
my_origin = None        # (위도, 경도)
my_dest = None          # (위도, 경도)

if my_origin and my_dest:
    rows = []
    for hour in HOURS:
        g = load_road_graph("hanam", modes=("drive",), speed_column=HOUR_TO_COLUMN[hour])
        p = shortest_path(g, my_origin, my_dest, algorithm="dijkstra")
        rows.append({"hour": hour, "소요_분": round(p.duration_s / 60, 2)})
    display(pd.DataFrame(rows))
else:
    print("[ ] my_origin 과 my_dest 를 채우세요")

## 정리

- 도로망 하나에 속도 컬럼이 여러 개 붙어 있고, `speed_column` 으로 골라 씁니다
- 시간대가 바뀌면 소요시간뿐 아니라 경로가 바뀝니다
- 시뮬레이터는 이 질의를 수십만 번 하므로 매번 다익스트라를 돌릴 수 없습니다.
  그래서 9장에서 소요시간을 예측하는 모델을 만듭니다
- 5장 실습에서는 대중교통 시간표 데이터로 넘어갑니다